(a) Consider a GPT-2 XL-sized model using our assignment architecture, which has the following 
configuration:
vocab_size:  50,257
context_length:  1,024
num_layers:  48
d_model:  1,600
num_heads:  25
d_ff:  4,288 (the nearest multiple of 64 to 8
3 × 1, 600)
Suppose we constructed our model using this configuration. How many trainable parameters 
would our model have? Assuming each parameter is represented using single-precision floating 
point, how much memory is required to just load this model?

number of parameters:
- token_embedding: 50257 * 1600
- transformer_block * nums_layers
    - Q,K,V projection:  1600 * 1600 * 3
    - output projection: 1600 * 1600
    - RMSnorm: 1600 *2
    - FFN: 1600 * 4288 * 3
- final_RMSNorm: 1600
- final-output: 1600 * 50257

total parameters: 1,640,452,800 ≈ 1.6B
memeory(single-float):  1,640,452,800×4=6,561,811,200 bytes
≈ 6.56 GB

(b) Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped 
model. How many FLOPs do these matrix multiplies require in total? Assume that our input 
sequence has context_length tokens.

total number of FLOPs :
T = context_length
d = d_model
f = d_ff
V = vocab_size

- transformer_block * num_layers
    - Q, K, V projection: [T, d] * [d, d*3]
    - SDPA: [T, d] * [d, T] + [T, T] * [T, d]
    - attention_output: [T, d] * [d, d]
    - FFN: [T, d] * [d, f] + [T, d] * [d, f] + [T, f] * [f, d]
- final_output = [T, d] * [d, V]

Total FLOPs: (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V = 3.5 TFOPs

In [12]:
T = 1024
d = 1600
f = 4288
V = 50257
num_layers = 48

total_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V
mha_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d) * num_layers
ffn_flops = 6*T*d*f * num_layers
final_flops = 2 * T * d * V

print(f'Total FLOPs: {total_flops/1e12} TFLOPs')
print(f'MHA layer requires FLOPs: {mha_flops/1e12} TFLOPs, up to {mha_flops/total_flops*100} %')
print(f'FFN layer requires FLOPs: {ffn_flops/1e12} TFLOPs, up to {ffn_flops/total_flops*100} %')
print(f'Final layer requires FLOPs: {final_flops/1e12} TFLOPs, up to {final_flops/total_flops*100} %')

Total FLOPs: 3.5167698944 TFLOPs
MHA layer requires FLOPs: 1.3287555072 TFLOPs, up to 37.78340770363938 %
FFN layer requires FLOPs: 2.0233322496 TFLOPs, up to 57.53382536690541 %
Final layer requires FLOPs: 0.1646821376 TFLOPs, up to 4.682766929455207 %


(c) Based on your analysis above, which parts of the model require the most FLOPs?

FFN layer required the most FLOPs

(d) Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium 
(24 layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As 
the model size increases, which parts of the Transformer LM take up proportionally more or 
less of the total FLOPs?
Deliverable: For each model, provide a breakdown of model components and its associated 
FLOPs (as a proportion of the total FLOPs required for a forward pass). In addition, provide 
a one-to-two sentence description of how varying the model size changes the proportional 
FLOPs of each component

In [8]:
T = 1024
d = 768
f = 2048
V = 50257
num_layers = 12

total_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V
mha_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d) * num_layers
ffn_flops = 6*T*d*f * num_layers
final_flops = 2 * T * d * V

print(f'Total FLOPs: {total_flops/1e9} GFLOPs')
print(f'MHA layer requires FLOPs: {mha_flops/1e9} GFLOPs, up to {mha_flops/total_flops*100} %')
print(f'FFN layer requires FLOPs: {ffn_flops/1e9} GFLOPs, up to {ffn_flops/total_flops*100} %')
print(f'Final layer requires FLOPs: {final_flops/1e9} GFLOPs, up to {final_flops/total_flops*100} %')

Total FLOPs: 291.6483072 GFLOPs
MHA layer requires FLOPs: 96.63676416 GFLOPs, up to 33.13469057570446 %
FFN layer requires FLOPs: 115.964116992 GFLOPs, up to 39.761628690845356 %
Final layer requires FLOPs: 79.047426048 GFLOPs, up to 27.10368073345018 %


In [9]:
T = 1024
d = 1024
f = 2752
V = 50257
num_layers = 24

total_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V
mha_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d) * num_layers
ffn_flops = 6*T*d*f * num_layers
final_flops = 2 * T * d * V

print(f'Total FLOPs: {total_flops/1e12} TFLOPs')
print(f'MHA layer requires FLOPs: {mha_flops/1e12} TFLOPs, up to {mha_flops/total_flops*100} %')
print(f'FFN layer requires FLOPs: {ffn_flops/1e12} TFLOPs, up to {ffn_flops/total_flops*100} %')
print(f'Final layer requires FLOPs: {final_flops/1e12} TFLOPs, up to {final_flops/total_flops*100} %')

Total FLOPs: 0.830172299264 TFLOPs
MHA layer requires FLOPs: 0.309237645312 TFLOPs, up to 37.24981495843196 %
FFN layer requires FLOPs: 0.415538085888 TFLOPs, up to 50.05443885039295 %
Final layer requires FLOPs: 0.105396568064 TFLOPs, up to 12.695746191175097 %


In [10]:
T = 1024
d = 1280
f = 3392
V = 50257
num_layers = 36

total_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V
mha_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d) * num_layers
ffn_flops = 6*T*d*f * num_layers
final_flops = 2 * T * d * V

print(f'Total FLOPs: {total_flops/1e12} TFLOPs')
print(f'MHA layer requires FLOPs: {mha_flops/1e12} TFLOPs, up to {mha_flops/total_flops*100} %')
print(f'FFN layer requires FLOPs: {ffn_flops/1e12} TFLOPs, up to {ffn_flops/total_flops*100} %')
print(f'Final layer requires FLOPs: {final_flops/1e12} TFLOPs, up to {final_flops/total_flops*100} %')

Total FLOPs: 1.76853090304 TFLOPs
MHA layer requires FLOPs: 0.67645734912 TFLOPs, up to 38.24967649460972 %
FFN layer requires FLOPs: 0.96032784384 TFLOPs, up to 54.30088002359773 %
Final layer requires FLOPs: 0.13174571008 TFLOPs, up to 7.449443481792539 %


随着模型 size 变化, MHA和LLN的FLOPs比例变化与模型正相关,而Final Layer负相关,LLN的变化相较于MHA更显著

(e) Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one 
forward pass change? How does the relative contribution of FLOPs of the model components 
change?

In [19]:
T = 1024 * 32
d = 1600
f = 4288
V = 50257
num_layers = 48

total_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d + 6*T*d*f) * num_layers + 2*T*d*V
mha_flops = (6*T*d*d + 4*T*T*d + 2*T*d*d) * num_layers
ffn_flops = 6*T*d*f * num_layers
final_flops = 2 * T * d * V

print(f'Total FLOPs: {total_flops/1e12} TFLOPs')
print(f'MHA layer requires FLOPs: {mha_flops/1e12} TFLOPs, up to {mha_flops/total_flops*100} %')
print(f'FFN layer requires FLOPs: {ffn_flops/1e12} TFLOPs, up to {ffn_flops/total_flops*100} %')
print(f'Final layer requires FLOPs: {final_flops/1e12} TFLOPs, up to {final_flops/total_flops*100} %')

Total FLOPs: 432.0822034432 TFLOPs
MHA layer requires FLOPs: 362.0657430528 TFLOPs, up to 83.7955694929231 %
FFN layer requires FLOPs: 64.7466319872 TFLOPs, up to 14.984794900424859 %
Final layer requires FLOPs: 5.2698284032 TFLOPs, up to 1.219635606652046 %


attention 会爆炸